# 06_rag_generation

Stage 6 of the pipeline: event card + retrieved passages -> prompt -> generator -> structured JSON.

All logic lives in `src/spacewx_rag/` (`generation`, `retrieval`, `cards`); this notebook shows what it does.
The long local-LLM runs were made from the command line so they could resume after an interruption:

```bash
python -m spacewx_rag generate --provider baseline
python -m spacewx_rag generate --provider ollama --run ollama_rag
python -m spacewx_rag generate --provider ollama --no-retrieval --run ollama_no_retrieval
python -m spacewx_rag generate --provider ollama --run ollama_consistency --events E216,E158,E039,E105,E221 --runs 3 --temperature 0.7
```
Results are saved one JSON file per event and run in `evaluation/results/generations/<run>/`. Set `RUN_LLM = True` below to regenerate here (about 3 to 5 minutes per event on an M2 laptop).

In [1]:
import json
import pandas as pd
from spacewx_rag.utils.config import Paths, load_params, load_env
from spacewx_rag.cards import load_cards
from spacewx_rag.retrieval import Retriever
from spacewx_rag.generation import (BaselineGenerator, EventDescription, generate_events, get_generator,
                                    load_results, system_prompt, user_prompt)
from spacewx_rag.generation.generators import constrained_schema

load_env()
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)
P, CFG = Paths(), load_params()
RUN_LLM = False
cards = load_cards()                      # evaluation_role removed (decision 32)
eval_ids = pd.read_csv(P.evaluation_set)["event_id"].tolist()
retriever = Retriever()
print("generation config:", CFG["generation"])

generation config: {'provider': 'ollama', 'model': 'qwen2.5:7b-instruct', 'temperature': 0.0, 'consistency_temperature': 0.7, 'consistency_runs': 3, 'num_ctx': 8192, 'num_predict': 1500, 'seed': 42, 'max_list_items': 6}


## Prompt design
The templates are in `prompts/`: `system.md` (the rules), `event_user.md` (card + passages), `judge_support.md` (used in notebook 07).

In [2]:
print(system_prompt())

You are a space weather analyst. You write a short, structured description of one past space weather event for engineers and satellite operators.

You get two inputs:
1. An EVENT CARD (JSON): the measured facts for the event, rule-based hints and data-quality caveats.
2. PASSAGES: background text from reference documents. Each passage starts with its id in square brackets, for example [d03_c001].

Rules. Follow every one.
1. Facts about the event come ONLY from the card. Copy numbers exactly as the card writes them, with their units. Do not calculate new numbers, do not round differently and do not convert units.
2. For every characteristic, list in "evidence" the card field paths that hold its numbers, written with dots, for example "dst.min_nT", "geomagnetic.kp_max", "drivers.bz_min_hourly_nT", "hints.3".
3. A physical interpretation or an impact is allowed ONLY if a passage says it. List the ids of those passages in "sources", written without brackets (for example "d03_c001"). Never

## Retrieval for one event
Several targeted queries per event (decision 41): physics, one per impact area for storms, index meaning, and provisional-versus-final Dst when the card says provisional. The May 2024 documents are only allowed for events in that window (decision 40).

In [3]:
c = cards["E216"]
passages = retriever.passages_for_event(c)
print(len(passages), "unique passages for E216")
pd.DataFrame([{"chunk_id": p["chunk_id"], "doc": p["doc_id"], "score": p["score"], "queries": ", ".join(p["queries"]),
               "text": p["text"][:90]} for p in passages])

/Users/manasstark/Projects/Spaceweather_GENAI_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1117 21:33:07.711000 10935 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7177.95it/s]

18 unique passages for E216


,chunk_id,doc,score,queries,text
0,d13_c009,13,0.783,physics,The storm compressed the Earth's subsolar magnetopause from higher than 10 R E down to 5 R
1,d03_c000,3,0.778,physics,A geomagnetic storm is a major disturbance of Earth's magnetosphere that occurs when there
2,d11_c002,11,0.761,physics,When the solar wind and the magnetosphere interact they operate as a system. The Earth's m
3,d07_c006,7,0.845,impacts: power systems,Assessing the impact of geomagnetic storms on the electrical power grid involves a number
4,d07_c008,7,0.829,impacts: power systems,. This means that a system that is near peak levels of demand prior to the geomagnetic sto
5,d03_c004,3,0.809,"impacts: satellites and spacecraft, impacts: GNSS navigation","During storms, the currents in the ionosphere, as well as the energetic particles that pre"
6,d15_c003,15,0.801,impacts: satellites and spacecraft,In addition to these long-term changes in upper atmospheric temperature and density caused
7,d13_c012,13,0.800,impacts: GNSS navigation,Agricultural users of John Deere RTK GPS equipment reported significantly degraded positio
8,d14_c002,14,0.832,impacts: HF radio,"Another type of space weather, the Radiation Storm caused by energetic solar protons, can"
9,d14_c001,14,0.820,impacts: HF radio,There are several types of space weather that can impact HF radio communication. In a typi


In [4]:
prompt = user_prompt(c, passages)
print(f"system + user prompt: {len(system_prompt()) + len(prompt):,} characters")
print(prompt[:1500], "...")

system + user prompt: 19,451 characters
EVENT CARD
{
 "event_id": "E216",
 "type": "storm",
 "class": "super",
 "window": {
  "start": "2024-05-10T07:00Z",
  "end": "2024-05-16T05:00Z",
  "hours": 143
 },
 "dst": {
  "min_nT": -406,
  "min_time": "2024-05-11T02:00Z",
  "first_below_minus50_time": "2024-05-10T19:00Z",
  "hours_from_minimum_to_back_above_minus30": 123,
  "minimum_is_final": false
 },
 "drivers": {
  "bz_min_hourly_nT": -39.3,
  "bz_min_time": "2024-05-10T21:00Z",
  "hours_bz_at_or_below_minus5": 23,
  "vbs_max_mV_per_m": 27.0,
  "vbs_mean_12h_before_dst_min_mV_per_m": 12.0,
  "speed_max_km_per_s": 1006,
  "density_max_per_cm3": 52.9,
  "dynamic_pressure_max_nPa_protons_only": 42.0,
  "total_field_max_nT": 71.6
 },
 "geomagnetic": {
  "kp_max": 9.0,
  "noaa_g_level_approx": "G5",
  "ap_max": 400
 },
 "solar": {
  "f107_mean_sfu": 222,
  "level": "high"
 },
 "hints": [
  "Super geomagnetic storm: Dst minimum -406 nT.",
  "Dst first fell below -50 nT 7 hours before reaching

## Safeguard: a constrained output schema
The output must match the pydantic model `EventDescription`. For each event the schema sent to Ollama also limits `evidence` to field paths that exist in the card and `sources` to the chunk ids that were retrieved, so the decoder cannot produce an invented citation. With no passages (the ablation), `sources` must be empty.

In [5]:
s = constrained_schema(c, passages)
print("evidence paths allowed:", len(s["$defs"]["Characteristic"]["properties"]["evidence"]["items"]["enum"]))
print("source ids allowed:", s["$defs"]["Impact"]["properties"]["sources"]["items"]["enum"])

evidence paths allowed: 44
source ids allowed: ['d13_c009', 'd03_c000', 'd11_c002', 'd07_c006', 'd07_c008', 'd03_c004', 'd15_c003', 'd13_c012', 'd14_c002', 'd14_c001', 'd12_c109', 'd12_c112', 'd13_c004', 'd20_c008', 'd12_c094', 'd21_c009', 'd21_c011', 'd21_c013']


## Generators
- `baseline`: offline and deterministic. The summary and characteristics are the card hints; interpretation and impacts are sentences copied verbatim from the top passage of each query. No model, no API key.
- `ollama`: `qwen2.5:7b-instruct` running locally (4-bit, about 4.7 GB), temperature 0, context 8,192 tokens.

In [6]:
if RUN_LLM:
    generate_events(eval_ids, get_generator("ollama"), retriever, run_name="ollama_rag")
runs = {name: load_results(name) for name in ["baseline", "ollama_rag", "ollama_no_retrieval", "ollama_consistency"]}
pd.DataFrame([{"run": k, "outputs": len(v), "valid JSON": sum(r["valid"] for r in v),
               "mean seconds": round(sum(r["seconds"] for r in v) / max(len(v), 1), 1),
               "mean prompt tokens": pd.Series([r.get("prompt_tokens") for r in v], dtype=float).mean(),
               "mean output tokens": pd.Series([r.get("output_tokens") for r in v], dtype=float).mean()}
              for k, v in runs.items()])

,run,outputs,valid JSON,mean seconds,mean prompt tokens,mean output tokens
0,baseline,15,15,0.0,NaN,NaN
1,ollama_rag,15,14,113.5,3638.4,608.266667
2,ollama_no_retrieval,15,15,64.9,1369.8,536.866667
3,ollama_consistency,15,15,73.8,3697.4,568.066667


## Example outputs

In [7]:
def show(run, eid):
    r = next(x for x in runs[run] if x["event_id"] == eid and x["run"] == 0)
    print(f"=== {run} / {eid} ({cards[eid]['class']}) valid={r['valid']} {r['seconds']}s")
    print(json.dumps(r["output"], indent=1, ensure_ascii=False) if r["valid"] else r["raw"][:1500])

show("ollama_rag", "E216")

=== ollama_rag / E216 (super) valid=True 281.7s
{
 "event_id": "E216",
 "summary": "The event, starting on May 10, 2024, and lasting for 143 hours, was characterized by a super geomagnetic storm with a Dst minimum of -406 nT, reaching an approximate NOAA G5 level. The storm was driven by a southward Bz component, fast solar wind, and high solar activity. The storm could have caused significant geoelectric fields and auroral displays, and may have impacted satellite drag and GPS accuracy, but no specific impacts were observed.",
 "characteristics": [
  {
   "statement": "Dst minimum -406 nT",
   "evidence": [
    "dst.min_nT"
   ]
  },
  {
   "statement": "Kp reached 9.0, indicating a G5-level geomagnetic storm",
   "evidence": [
    "geomagnetic.kp_max",
    "geomagnetic.noaa_g_level_approx"
   ]
  },
  {
   "statement": "Sustained southward Bz: 23 hours at or below -5 nT, with a minimum hourly Bz of -39.3 nT",
   "evidence": [
    "drivers.bz_min_hourly_nT",
    "drivers.bz_min_time",

In [8]:
show("baseline", "E216")

=== baseline / E216 (super) valid=True 0.001s
{
 "event_id": "E216",
 "summary": "Super geomagnetic storm: Dst minimum -406 nT. Kp reached 9.0 (approximately NOAA G5 level). Solar activity high (mean F10.7 222 sfu).",
 "characteristics": [
  {
   "statement": "Super geomagnetic storm: Dst minimum -406 nT.",
   "evidence": [
    "hints.0"
   ]
  },
  {
   "statement": "Dst first fell below -50 nT 7 hours before reaching its minimum.",
   "evidence": [
    "hints.1"
   ]
  },
  {
   "statement": "Dst was back above -30 nT 123 hours after the minimum.",
   "evidence": [
    "hints.2"
   ]
  },
  {
   "statement": "Kp reached 9.0 (approximately NOAA G5 level).",
   "evidence": [
    "hints.3"
   ]
  },
  {
   "statement": "Sustained southward Bz: 23 hours at or below -5 nT; minimum hourly Bz -39.3 nT.",
   "evidence": [
    "hints.4"
   ]
  },
  {
   "statement": "Strongly southward Bz (hourly mean at or below -20 nT).",
   "evidence": [
    "hints.5"
   ]
  },
  {
   "statement": "Very fa

In [9]:
show("ollama_rag", "E143")     # quiet window with one brief southward-Bz hour

=== ollama_rag / E143 (quiet) valid=True 73.2s
{
 "event_id": "E143",
 "summary": "During the period from November 18 to 19, 2022, the Dst index ranged from -6 nT to above -30 nT, indicating a quiet event with minimal geomagnetic disturbance. The Bz component was southward for one hour, and the solar wind speed was ordinary, with a maximum of 380 km/s.",
 "characteristics": [
  {
   "statement": "Dst index ranged from -6 nT to above -30 nT.",
   "evidence": [
    "dst.min_nT",
    "dst.hours_from_minimum_to_back_above_minus30"
   ]
  },
  {
   "statement": "Bz component was southward for one hour, with a minimum hourly value of -7.1 nT.",
   "evidence": [
    "drivers.bz_min_hourly_nT",
    "drivers.bz_min_time"
   ]
  },
  {
   "statement": "Solar wind speed was ordinary, with a maximum of 380 km/s.",
   "evidence": [
    "drivers.speed_max_km_per_s"
   ]
  }
 ],
 "possible_physical_interpretation": [
  {
   "statement": "Southward Bz is the main way solar wind energy enters the magne

In [10]:
show("ollama_no_retrieval", "E216")   # card only, no passages

=== ollama_no_retrieval / E216 (super) valid=True 70.6s
{
 "event_id": "E216",
 "summary": "The event, lasting 143 hours, featured a super geomagnetic storm with a Dst minimum of -406 nT, reaching an approximate NOAA G5 level. The southward Bz component was sustained for 23 hours, and the solar wind was very fast with a maximum speed of 1006 km/s.",
 "characteristics": [
  {
   "statement": "Dst minimum -406 nT",
   "evidence": [
    "dst.min_nT"
   ]
  },
  {
   "statement": "Kp reached 9.0, indicating a G5 level storm",
   "evidence": [
    "geomagnetic.kp_max",
    "geomagnetic.noaa_g_level_approx"
   ]
  },
  {
   "statement": "Southward Bz component was sustained for 23 hours, with a minimum hourly value of -39.3 nT",
   "evidence": [
    "drivers.bz_min_hourly_nT",
    "drivers.bz_min_time",
    "drivers.hours_bz_at_or_below_minus5"
   ]
  },
  {
   "statement": "Solar wind was very fast, with a maximum speed of 1006 km/s",
   "evidence": [
    "drivers.speed_max_km_per_s"
   ]
 

### 06 RAG generation findings

- Pipeline: card (without `evaluation_role`) + de-duplicated passages from several targeted queries (physics, one per impact area for storms, index meaning, provisional Dst) -> `prompts/system.md` rules -> JSON checked against the pydantic schema. Storm prompts are about 3,700 to 4,900 tokens (15 to 18 passages); quiet ones about 2,600.
- Safeguard: a per-event schema limits `evidence` to real card fields and `sources` to the retrieved chunk ids. No output in any run cited a chunk that wasn't retrieved, and every evidence path existed.
- Local model `qwen2.5:7b-instruct` (Ollama, temperature 0): 14 of 15 outputs valid. E039 got stuck repeating one evidence path until the 1,500-token limit and its JSON was cut off; `max_list_items: 6` now prevents that loop (decision 55), but the recorded run is kept as it was.
- Speed on an M2 laptop: 63 to 85 s per storm event with Low Power Mode off, 150 to 280 s with it on (the first four events). Quiet events about 70 s.
- What the model does well: it copies card numbers correctly (no number errors in any LLM run), phrases impacts as "could"/"may", repeats the provisional-Dst and OMNI caveats, and leaves `potential_impacts` empty for all five quiet windows.
- What it does badly: it often leaves out Kp and the G level (7 of 14 outputs) and sometimes the storm class; it puts impact language into the summary (E216, E231, E117); it applied G5 effects to the G4 storm E231 because only the G5 row of the NOAA scales was retrieved; it misread an auroral elevation angle (55 degrees) as a latitude; and it often lists every topic as "no supporting source" even when it gave sourced impacts (1 of 14 flagged; the flag is conservative).
- Without passages (ablation) the same model still writes a general physics line ("southward Bz is the main way energy enters the magnetosphere") for nearly every event, with no source. With passages it cites d03_c000 or d11_c002 for that line.
- The offline baseline is valid for all 15 events by construction and needs no model, but its copied impact sentences can be off-topic (for example a Hanle aurora report for E216's aurora line).